# Fase 1: Definición del Problema Analítico, Entorno Reproducible y Validación del Dataset
**Curso:** MCDI500 — Programación para la Ciencia de Datos (Grupo 3)  
**Docente:** Dr. Omar Iván Salinas Silva  
**Integrantes:** Madeleine Aguilera, Victoria Álvarez, Ignacio Peña y Raúl Hormazábal

---

## 1. Situación Problematizadora, Pregunta Analizable y Objetivo General Analítico
Los establecimientos que generan residuos peligrosos sobre los umbrales del **D.S. N° 148/2003 (MINSAL)** deben declararlos anualmente en el **Sistema de Declaración y Seguimiento de Residuos Peligrosos (SIDREP)** del **Registro de Emisiones y Transferencias de Contaminantes (RETC)**. Al consolidar los registros públicos de **2023, 2024 y 2025** (`Base_Maestra_2023_2025.xlsx`, con `118.280 filas × 21 columnas`), se requiere distinguir qué establecimientos conforman el núcleo permanente de generación frente a los declarantes intermitentes:

- **Pregunta del proyecto:** *¿Qué establecimientos (`id_vu`) se mantienen como generadores de residuos peligrosos entre 2023, 2024 y 2025, cuánto varía su volumen (`cantidad_kg`) año a año, y qué parte del cambio total se explica por establecimientos que entran o salen del registro?*
- **Objetivo General Analítico:** *Determinar la persistencia interanual de los establecimientos generadores de residuos peligrosos en Chile durante el trienio 2023–2025 y cuantificar el peso relativo del núcleo persistente frente a la entrada y salida de establecimientos en la variación del volumen total declarado, según región y rubro productivo.*


In [1]:
import os
import sys
from pathlib import Path

# Posicionar el directorio de trabajo en la raíz del repositorio
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent.parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

print("Intérprete activo:", sys.executable)
print("Raíz del proyecto:", Path.cwd())

for lib in ["numpy", "pandas", "matplotlib", "seaborn", "sklearn", "openpyxl"]:
    print(f" - {lib:12}: {__import__(lib).__version__}")


Intérprete activo: C:\Users\Usuario\proyecto-grupo3-mcdi500\.venv\Scripts\python.exe
Raíz del proyecto: C:\Users\Usuario\proyecto-grupo3-mcdi500
 - numpy       : 2.5.4
 - pandas      : 3.1.0
 - matplotlib  : 3.11.2
 - seaborn     : 0.13.2
 - sklearn     : 1.9.1
 - openpyxl    : 3.1.5


## 2. Carga Segura del Dataset Original desde `src/carga.py`
Siguiendo el principio de inmutabilidad de los datos crudos, el archivo `F1/data/raw/Base_Maestra_2023_2025.xlsx` se lee de forma controlada mediante `cargar_datos_seguro()` sin modificarlo y se mantiene excluido del repositorio remoto por contener 398 RUT de personas naturales en `rut_razon_social`.


In [2]:
import pandas as pd
from IPython.display import display
from src.carga import COLUMNAS_ESPERADAS_RETC, cargar_datos_seguro

RUTA_RAW = Path("F1/data/raw/Base_Maestra_2023_2025.xlsx")
df_raw = cargar_datos_seguro(RUTA_RAW, columnas_esperadas=COLUMNAS_ESPERADAS_RETC)
print(f"[OK] Dataset cargado correctamente desde: {RUTA_RAW}")
print(f"Dimensiones originales: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
display(df_raw.head(3))


[OK] Dataset cargado correctamente desde: F1\data\raw\Base_Maestra_2023_2025.xlsx
Dimensiones originales: 118,280 filas × 21 columnas


,año,id_vu,rut_razon_social,id_ciiu4,ciiu4,id_ciiu6,ciiu6,id_rubro,rubro,codigo_territorial,...,provincia,region,latitud,longitud,cantidad_kg,id_peligrosidad,peligrosidad,id_lista_a,id_estado_materia,estado_materia
0,2023,5,94283000-9,C2599,Fabricación de otros productos elaborados de m...,C259900,Fabricación de otros productos elaborados de m...,10,Otras industrias manufactureras,15101,...,Arica,Arica y Parinacota,-18.473556,-70.317359,24750.0,"2,",TOXICO CRONICO,A3020,2,líquido
1,2023,6,80125000-9,C2394,"Fabricación de cemento, cal y yeso",C239400,"Fabricación de cemento, cal y yeso",8,"Producción de cemento, cal y yeso",15101,...,Arica,Arica y Parinacota,-18.466515,-70.295403,15.0,"3,5,",TOXICO POR LIXIVIACION##INFLAMABLE,A4130,1,sólido
2,2023,6,80125000-9,C2394,"Fabricación de cemento, cal y yeso",C239400,"Fabricación de cemento, cal y yeso",8,"Producción de cemento, cal y yeso",15101,...,Arica,Arica y Parinacota,-18.466515,-70.295403,28.0,"3,",TOXICO POR LIXIVIACION,A3020,1,sólido


## 3. Diccionario y Perfilamiento de las 21 Variables (Roles Analíticos Unificados)
Se construye el perfil técnico de las 21 columnas declarando de manera consistente el rol analítico de cada variable (en particular, `id_lista_a` como categórica de alta cardinalidad con `59` códigos que se imputará por moda de `id_rubro`, y `estado_materia` como categórica nominal de `4` categorías que se codificará mediante One-Hot).


In [3]:
DICCIONARIO_VARIABLES = {
    "año":                ("fecha",             "Año calendario de la declaración RETC (2023, 2024, 2025)"),
    "id_vu":              ("identificador",     "Identificador numérico del establecimiento en Ventanilla Única"),
    "rut_razon_social":   ("alta cardinalidad", "RUT de la organización declarante (se anonimiza por frecuencia en F2/F3)"),
    "id_ciiu4":           ("alta cardinalidad", "Código CIIU4 de actividad económica a 4 dígitos (546 códigos)"),
    "ciiu4":              ("texto libre",       "Glosa descriptiva de la actividad económica según CIIU4"),
    "id_ciiu6":           ("alta cardinalidad", "Código CIIU6 de actividad económica a 6 dígitos (388 códigos)"),
    "ciiu6":              ("texto libre",       "Glosa descriptiva detallada de la actividad económica según CIIU6"),
    "id_rubro":           ("discreta",          "Código entero del rubro económico RETC (0 a 20)"),
    "rubro":              ("nominal",           "Categoría de rubro industrial o sector productivo (19 rubros)"),
    "codigo_territorial": ("discreta",          "Código Único Territorial (CUT) comunal entero (380 códigos)"),
    "comuna":             ("alta cardinalidad", "Nombre de la comuna del establecimiento generador (344 comunas)"),
    "provincia":          ("nominal",           "Nombre de la provincia administrativa (58 provincias)"),
    "region":             ("nominal",           "Región administrativa de Chile (16 regiones; 19 etiquetas en crudo)"),
    "latitud":            ("continua",          "Latitud geográfica del establecimiento en grados decimales"),
    "longitud":           ("continua",          "Longitud geográfica del establecimiento en grados decimales"),
    "cantidad_kg":        ("continua",          "Masa anual declarada de residuo peligroso en kilogramos (kg)"),
    "id_peligrosidad":    ("nominal",           "Códigos de característica de peligrosidad D.S. N° 148/2003"),
    "peligrosidad":       ("texto libre",       "Glosa de característica(s) de peligrosidad declaradas"),
    "id_lista_a":         ("alta cardinalidad", "Código Lista A D.S. 148/2003 (59 códigos; se imputa por moda de id_rubro y se codifica por frecuencia)"),
    "id_estado_materia":  ("discreta",          "Código administrativo RETC de estado físico (1=sólido, 2=líquido, 3=gaseoso, 4=lodo)"),
    "estado_materia":     ("nominal",           "Estado físico del residuo (sólido, líquido, lodo, gaseoso; se codifica con One-Hot)"),
}

perfil_f1 = pd.DataFrame([
    {
        "variable": col,
        "dtype_pandas": str(df_raw[col].dtype),
        "rol_analitico": DICCIONARIO_VARIABLES[col][0],
        "valores_unicos": int(df_raw[col].nunique(dropna=True)),
        "nulos_n": int(df_raw[col].isna().sum()),
        "nulos_pct": round(df_raw[col].isna().mean() * 100, 2),
        "descripcion": DICCIONARIO_VARIABLES[col][1],
    }
    for col in df_raw.columns
])
display(perfil_f1)


,variable,dtype_pandas,rol_analitico,valores_unicos,nulos_n,nulos_pct,descripcion
0,año,int64,fecha,3,0,0.00,"Año calendario de la declaración RETC (2023, 2..."
1,id_vu,int64,identificador,12192,0,0.00,Identificador numérico del establecimiento en ...
2,rut_razon_social,str,alta cardinalidad,6779,0,0.00,RUT de la organización declarante (se anonimiz...
3,id_ciiu4,str,alta cardinalidad,546,1,0.00,Código CIIU4 de actividad económica a 4 dígito...
4,ciiu4,str,texto libre,335,1,0.00,Glosa descriptiva de la actividad económica se...
5,id_ciiu6,str,alta cardinalidad,388,1,0.00,Código CIIU6 de actividad económica a 6 dígito...
6,ciiu6,str,texto libre,388,1,0.00,Glosa descriptiva detallada de la actividad ec...
7,id_rubro,int64,discreta,20,0,0.00,Código entero del rubro económico RETC (0 a 20)
8,rubro,str,nominal,19,1,0.00,Categoría de rubro industrial o sector product...
9,codigo_territorial,int64,discreta,380,0,0.00,Código Único Territorial (CUT) comunal entero ...


## 4. Verificación Programática de Umbrales (`assert`) y Exportación del Reporte de Fase 1
Se comprueba mediante aserciones que la base supera los requisitos de volumen, dimensionalidad, diversidad de roles y proporción de datos faltantes, exportando el resumen en `F1/docs/informe_dataset.md`.


In [4]:
max_pct_nulos = float(perfil_f1["nulos_pct"].max())
duplicados_n = int(df_raw.duplicated().sum())

assert df_raw.shape[0] >= 2000, "El dataset debe tener al menos 2.000 filas"
assert df_raw.shape[1] >= 12, "El dataset debe tener al menos 12 columnas"
assert 1.0 <= max_pct_nulos <= 60.0, "El porcentaje máximo de nulos debe estar entre 1% y 60%"

ruta_informe = Path("F1/docs/informe_dataset.md")
ruta_informe.parent.mkdir(parents=True, exist_ok=True)
lineas_md = [
    "# Informe de Validación del Dataset (Fase 1) — Grupo 3 MCDI500",
    f"- **Archivo evaluado:** `{RUTA_RAW}`",
    f"- **Dimensiones:** {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas",
    f"- **Filas duplicadas exactas:** {duplicados_n}",
    f"- **Porcentaje máximo de valores faltantes:** {max_pct_nulos}% (`id_lista_a`: 4.764 nulos)",
    "",
    "## Diccionario y Perfil de Variables",
    "| " + " | ".join(perfil_f1.columns) + " |",
    "| " + " | ".join(["---"] * len(perfil_f1.columns)) + " |",
]
for fila in perfil_f1.values:
    lineas_md.append("| " + " | ".join(str(v) for v in fila) + " |")
ruta_informe.write_text("\n".join(lineas_md) + "\n", encoding="utf-8")

print(f"[OK] Criterios verificados -> Filas: {df_raw.shape[0]:,} | Duplicados: {duplicados_n} | Máximo nulos: {max_pct_nulos}%")
print(f"[OK] Reporte exportado en: {ruta_informe}")


[OK] Criterios verificados -> Filas: 118,280 | Duplicados: 118 | Máximo nulos: 4.03%
[OK] Reporte exportado en: F1\docs\informe_dataset.md


## 5. Auditoría de Control de Versiones (Git) y Cierre de Fase 1
Se consulta programáticamente el historial reciente de commits del repositorio para dejar evidencia reproducible del flujo colaborativo.


In [5]:
import subprocess

res_rama = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True, encoding="utf-8")
res_log = subprocess.run(["git", "log", "--oneline", "-8"], capture_output=True, text=True, encoding="utf-8")
print("Rama activa:", res_rama.stdout.strip())
print("\nÚltimos commits del repositorio:")
print(res_log.stdout.strip())


Rama activa: feat/notebooks-f1-f2-f3

Últimos commits del repositorio:
32c98de style: restaura indentacion PEP8 en modulos del paquete src
efefde5 Merge pull request #5 from Rulman75/feat/eficiencia-y-recursividad
5c7d4ad feat: implementa mediciones timeit/tracemalloc, recursividad CIIU y paquete compartido src
1989ce3 Merge pull request #4 from Rulman75/feat/pipeline-observer
fcb93d8 feat: implementa clase Pipeline polimorfica y BitacoraObserver con patron Observer
fa272ac Merge pull request #3 from Rulman75/feat/transformadores-y-strategy
19cdbb2 feat: implementa clases hijas ImputadorPorGrupo, CodificadorCategorias, EscaladorRobusto y patron Strategy
97619f2 Merge pull request #2 from Rulman75/feat/limpieza-y-clase-base
